# Phase 2: Data Preprocessing & Feature Engineering

## Overview
This notebook implements comprehensive preprocessing for hospital readmission prediction:
- Drop duplicate patient encounters (keep first per patient)
- Drop columns with excessive missing values (weight, payer_code)
- Encode age as ordinal (already in bracket format)
- Map admission/discharge/source IDs using IDS_mapping.csv
- One-hot encode categorical variables
- Encode 23 medication columns as change indicators
- Apply SMOTE to handle class imbalance
- Train/validation/test split: 70/15/15

## Key Processing Steps
1. Load data and mappings
2. Drop duplicates by patient_nbr (keep first)
3. Remove high missing columns
4. Create binary target (readmitted within 30 days)
5. Map ID columns using lookup tables
6. Encode age brackets as ordinal values
7. Process medication columns as change indicators
8. One-hot encode remaining categoricals
9. Apply SMOTE to training data only
10. Scale numeric features
11. Create train/validation/test splits

In [7]:
# Imports and setup
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
import sys
sys.path.append('../src')
from preprocess import load_data, create_binary_target

warnings.filterwarnings('ignore')
print("Phase 2: Data Preprocessing & Feature Engineering")

Phase 2: Data Preprocessing & Feature Engineering


In [8]:
# Load data and mappings
data_path = '../diabetes+130-us+hospitals+for+years+1999-2008/diabetic_data.csv'
mapping_path = '../diabetes+130-us+hospitals+for+years+1999-2008/IDS_mapping.csv'

df, df_mapping = load_data(data_path, mapping_path)
print(f"Loaded data shape: {df.shape}")
print(f"Mapping shape: {df_mapping.shape}")
print(f"\nFirst few rows of data:")
print(df.head())
print(f"\nColumn names: {df.columns.tolist()}")

Loaded data shape: (101766, 50)
Mapping shape: (67, 2)

First few rows of data:
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   
3        500364     82442376        Caucasian    Male  [30-40)      ?   
4         16680     42519267        Caucasian    Male  [40-50)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburid

In [9]:
# Step 1: Drop duplicate patient encounters (keep first encounter per patient)
print("Step 1: Dropping duplicate patient encounters")
print(f"Original shape: {df.shape}")
print(f"Unique patients: {df['patient_nbr'].nunique()}")

df = df.drop_duplicates(subset=['patient_nbr'], keep='first')
print(f"After dropping duplicates: {df.shape}")
print(f"Unique patients: {df['patient_nbr'].nunique()}")

Step 1: Dropping duplicate patient encounters
Original shape: (101766, 50)
Unique patients: 71518
After dropping duplicates: (71518, 50)
Unique patients: 71518


In [10]:
# Step 2: Drop columns with excessive missing values and unnecessary columns
print("\nStep 2: Dropping high missing-value and unnecessary columns")
print(f"Before: {df.shape}")

# Drop weight and payer_code (too many missing values)
cols_to_drop = ['weight', 'payer_code']
df = df.drop(columns=cols_to_drop, errors='ignore')

# Also drop columns with >90% missing values
missing_pct = (df.isnull().sum() / len(df)) * 100
cols_high_missing = missing_pct[missing_pct > 90].index.tolist()
df = df.drop(columns=cols_high_missing, errors='ignore')

print(f"After: {df.shape}")
print(f"Dropped columns: {cols_to_drop + cols_high_missing}")

# Step 3: Create binary target variable (readmitted within 30 days)
print("\nStep 3: Creating binary target variable")
df = create_binary_target(df)
print(f"Target variable distribution:\n{df['readmitted_30'].value_counts()}")
print(f"Class imbalance ratio: {df['readmitted_30'].value_counts()[0] / df['readmitted_30'].value_counts()[1]:.2f}:1")


Step 2: Dropping high missing-value and unnecessary columns
Before: (71518, 50)
After: (71518, 47)
Dropped columns: ['weight', 'payer_code', 'max_glu_serum']

Step 3: Creating binary target variable
Target variable distribution:
readmitted_30
0    65225
1     6293
Name: count, dtype: int64
Class imbalance ratio: 10.36:1


In [11]:
# Step 4: Map ID columns using IDS_mapping.csv
print("\nStep 4: Mapping ID columns")
print("Mapping columns: admission_type_id, discharge_disposition_id, admission_source_id")

# The IDS_mapping.csv has a mixed format with multiple ID types
# Parse it to extract separate mappings for each ID column
mapping_dict_admission_type = {}
mapping_dict_discharge = {}
mapping_dict_admission_source = {}

with open('../diabetes+130-us+hospitals+for+years+1999-2008/IDS_mapping.csv', 'r') as f:
    lines = f.readlines()
    
current_mapping = None
for i, line in enumerate(lines):
    line = line.strip()
    if not line or line == ',':
        continue
    
    # Check if this line indicates a new mapping type
    if 'admission_type_id' in line and 'description' in line:
        current_mapping = 'admission_type'
        continue
    elif 'discharge_disposition_id' in line and 'description' in line:
        current_mapping = 'discharge'
        continue
    elif 'admission_source_id' in line and 'description' in line:
        current_mapping = 'admission_source'
        continue
    
    # Parse the ID and description
    parts = line.split(',', 1)
    if len(parts) == 2:
        try:
            id_val = int(parts[0].strip())
            desc = parts[1].strip()
            
            if current_mapping == 'admission_type':
                mapping_dict_admission_type[id_val] = desc
            elif current_mapping == 'discharge':
                mapping_dict_discharge[id_val] = desc
            elif current_mapping == 'admission_source':
                mapping_dict_admission_source[id_val] = desc
        except (ValueError, IndexError):
            pass

print(f"  admission_type_id: {len(mapping_dict_admission_type)} mappings")
print(f"  discharge_disposition_id: {len(mapping_dict_discharge)} mappings")
print(f"  admission_source_id: {len(mapping_dict_admission_source)} mappings")

# Apply mappings to dataframe
if 'admission_type_id' in df.columns and mapping_dict_admission_type:
    df['admission_type_id_desc'] = df['admission_type_id'].map(mapping_dict_admission_type)
    
if 'discharge_disposition_id' in df.columns and mapping_dict_discharge:
    df['discharge_disposition_id_desc'] = df['discharge_disposition_id'].map(mapping_dict_discharge)
    
if 'admission_source_id' in df.columns and mapping_dict_admission_source:
    df['admission_source_id_desc'] = df['admission_source_id'].map(mapping_dict_admission_source)

# Display sample of mapped columns
print("\nSample of mapped columns:")
if 'admission_type_id_desc' in df.columns:
    print(df[['admission_type_id', 'admission_type_id_desc']].head(5))


Step 4: Mapping ID columns
Mapping columns: admission_type_id, discharge_disposition_id, admission_source_id
  admission_type_id: 8 mappings
  discharge_disposition_id: 30 mappings
  admission_source_id: 25 mappings

Sample of mapped columns:
   admission_type_id admission_type_id_desc
0                  6                   NULL
1                  1              Emergency
2                  1              Emergency
3                  1              Emergency
4                  1              Emergency


In [12]:
# Step 5: Encode age as ordinal (age is already in bracket format)
print("\nStep 5: Encoding age as ordinal")
print(f"Age values: {df['age'].unique()}")

# Age is in bracket format, encode as ordinal
age_mapping = {
    '[0-10)': 0,
    '[10-20)': 1,
    '[20-30)': 2,
    '[30-40)': 3,
    '[40-50)': 4,
    '[50-60)': 5,
    '[60-70)': 6,
    '[70-80)': 7,
    '[80-90)': 8,
    '[90-100)': 9
}

df['age_ordinal'] = df['age'].map(age_mapping)
print(f"Age ordinal mapping:\n{age_mapping}")
print(f"Age ordinal values:\n{df['age_ordinal'].value_counts().sort_index()}")

# We can drop the original age column as we now have the ordinal encoding
df = df.drop(columns=['age'], errors='ignore')
print("Original 'age' column dropped")


Step 5: Encoding age as ordinal
Age values: <ArrowStringArray>
[  '[0-10)',  '[10-20)',  '[20-30)',  '[30-40)',  '[40-50)',  '[50-60)',
  '[60-70)',  '[70-80)',  '[80-90)', '[90-100)']
Length: 10, dtype: str
Age ordinal mapping:
{'[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3, '[40-50)': 4, '[50-60)': 5, '[60-70)': 6, '[70-80)': 7, '[80-90)': 8, '[90-100)': 9}
Age ordinal values:
age_ordinal
0      154
1      535
2     1127
3     2699
4     6878
5    12466
6    15960
7    18210
8    11589
9     1900
Name: count, dtype: int64
Original 'age' column dropped


In [13]:
# Step 6: Encode 23 medication columns as change indicators
print("\nStep 6: Processing medication columns as change indicators")

# Medication columns have values: No/Steady/Up/Down
# Encode as change indicators: No=0 (no change), Steady=0 (no change), Up=1 (change), Down=1 (change)
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'metformin-rosiglitazone', 'metformin-pioglitazone', 'change'
]

# Find which medication columns actually exist
existing_med_cols = [col for col in medication_cols if col in df.columns]
print(f"Found {len(existing_med_cols)} medication columns")

# Encode each medication column: No/Steady -> 0 (no change), Up/Down -> 1 (change)
for col in existing_med_cols:
    # Check unique values
    print(f"  {col}: {df[col].unique()}")
    
    # Encode as change indicator
    change_mapping = {
        'No': 0,
        'Steady': 0,
        'Up': 1,
        'Down': 1
    }
    
    # Handle NaN values
    df[col] = df[col].fillna('No')
    df[col] = df[col].map(change_mapping)
    
    # Rename to indicate it's a change indicator
    if col != 'change':
        df.rename(columns={col: f'{col}_change'}, inplace=True)

print(f"Processed {len(existing_med_cols)} medication columns as change indicators")


Step 6: Processing medication columns as change indicators
Found 23 medication columns
  metformin: <ArrowStringArray>
['No', 'Steady', 'Up', 'Down']
Length: 4, dtype: str
  repaglinide: <ArrowStringArray>
['No', 'Up', 'Steady', 'Down']
Length: 4, dtype: str
  nateglinide: <ArrowStringArray>
['No', 'Steady', 'Down', 'Up']
Length: 4, dtype: str
  chlorpropamide: <ArrowStringArray>
['No', 'Steady', 'Down', 'Up']
Length: 4, dtype: str
  glimepiride: <ArrowStringArray>
['No', 'Steady', 'Down', 'Up']
Length: 4, dtype: str
  acetohexamide: <ArrowStringArray>
['No', 'Steady']
Length: 2, dtype: str
  glipizide: <ArrowStringArray>
['No', 'Steady', 'Up', 'Down']
Length: 4, dtype: str
  glyburide: <ArrowStringArray>
['No', 'Steady', 'Up', 'Down']
Length: 4, dtype: str
  tolbutamide: <ArrowStringArray>
['No', 'Steady']
Length: 2, dtype: str
  pioglitazone: <ArrowStringArray>
['No', 'Steady', 'Up', 'Down']
Length: 4, dtype: str
  rosiglitazone: <ArrowStringArray>
['No', 'Steady', 'Up', 'Down']
Len

In [14]:
# Step 7: One-hot encode categorical variables
print("\nStep 7: One-hot encoding categorical variables")

# Identify categorical columns (object dtype and not already processed)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns to encode: {categorical_cols}")

# Exclude target variable from encoding
if 'readmitted' in categorical_cols:
    categorical_cols.remove('readmitted')

# One-hot encode categorical columns
if categorical_cols:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print(f"One-hot encoded {len(categorical_cols)} categorical columns")
    print(f"New shape after encoding: {df.shape}")
else:
    print("No categorical columns to encode")

# Display final feature info
print(f"\nFinal features: {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")


Step 7: One-hot encoding categorical variables
Categorical columns to encode: ['race', 'gender', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'A1Cresult', 'glimepiride-pioglitazone', 'diabetesMed', 'readmitted', 'admission_type_id_desc', 'discharge_disposition_id_desc', 'admission_source_id_desc']
One-hot encoded 12 categorical columns
New shape after encoding: (71518, 2346)

Final features: 2346 columns
Columns: ['encounter_id', 'patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'metformin_change', 'repaglinide_change', 'nateglinide_change', 'chlorpropamide_change', 'glimepiride_change', 'acetohexamide_change', 'glipizide_change', 'glyburide_change', 'tolbutamide_change', 'pioglitazone_change', 'rosiglitazone_change', 'acarbose_change', 'miglitol_change', 'troglitazone_change', 'tolaza

In [15]:
# Step 8: Prepare data for splitting (separate features and target)
print("\nStep 8: Preparing features and target")

# Separate features and target
X = df.drop(columns=['readmitted_30', 'readmitted', 'patient_nbr'], errors='ignore')
y = df['readmitted_30']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# Step 9: Train/validation/test split: 70/15/15
print("\nStep 9: Train/Validation/Test split (70/15/15)")

# First split: 70% train, 30% temp (for val and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: split temp into 50% validation, 50% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nTarget distribution in train set:")
print(y_train.value_counts())
print(f"\nTarget distribution in validation set:")
print(y_val.value_counts())
print(f"\nTarget distribution in test set:")
print(y_test.value_counts())


Step 8: Preparing features and target
Features shape: (71518, 2343)
Target shape: (71518,)
Target distribution:
readmitted_30
0    65225
1     6293
Name: count, dtype: int64

Step 9: Train/Validation/Test split (70/15/15)
Train set: 50062 samples (70.0%)
Validation set: 10728 samples (15.0%)
Test set: 10728 samples (15.0%)

Target distribution in train set:
readmitted_30
0    45657
1     4405
Name: count, dtype: int64

Target distribution in validation set:
readmitted_30
0    9784
1     944
Name: count, dtype: int64

Target distribution in test set:
readmitted_30
0    9784
1     944
Name: count, dtype: int64


In [17]:
# Step 10: Handle missing values and apply SMOTE to training data
print("\nStep 10: Handling missing values and applying SMOTE")

# Check for NaN values
print(f"NaN values in X_train:")
nan_counts = X_train.isnull().sum()
if nan_counts.sum() > 0:
    print(nan_counts[nan_counts > 0])
else:
    print("  No NaN values")

# Impute missing values in training and other sets
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')

# Fit on training data and transform all sets
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_val = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

print(f"NaN values after imputation: {X_train.isnull().sum().sum()}")

# Apply SMOTE to training data only
print(f"\nTraining set before SMOTE:")
print(f"  Class 0: {(y_train == 0).sum()}, Class 1: {(y_train == 1).sum()}")
print(f"  Ratio: {(y_train == 0).sum() / (y_train == 1).sum():.2f}:1")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\nTraining set after SMOTE:")
print(f"  Class 0: {(y_train_smote == 0).sum()}, Class 1: {(y_train_smote == 1).sum()}")
print(f"  Ratio: {(y_train_smote == 0).sum() / (y_train_smote == 1).sum():.2f}:1")
print(f"  New shape: {X_train_smote.shape}")

# Convert back to DataFrame to preserve column names
X_train_smote = pd.DataFrame(X_train_smote, columns=X_train.columns)
y_train_smote = pd.Series(y_train_smote, name='readmitted_30')


Step 10: Handling missing values and applying SMOTE
NaN values in X_train:
change    22416
dtype: int64
NaN values after imputation: 0

Training set before SMOTE:
  Class 0: 45657, Class 1: 4405
  Ratio: 10.36:1

Training set after SMOTE:
  Class 0: 45657, Class 1: 45657
  Ratio: 1.00:1
  New shape: (91314, 2343)


In [18]:
# Step 11: Scale numeric features using training data statistics
print("\nStep 11: Scaling numeric features")

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f"Scaling complete. Feature means and stds:")
print(f"  Train: mean={X_train_scaled.mean().mean():.4f}, std={X_train_scaled.std().mean():.4f}")
print(f"  Val:   mean={X_val_scaled.mean().mean():.4f}, std={X_val_scaled.std().mean():.4f}")
print(f"  Test:  mean={X_test_scaled.mean().mean():.4f}, std={X_test_scaled.std().mean():.4f}")


Step 11: Scaling numeric features
Scaling complete. Feature means and stds:
  Train: mean=0.0000, std=0.9304
  Val:   mean=0.0012, std=0.8602
  Test:  mean=0.0013, std=0.8633


In [19]:
# Step 12: Save preprocessed datasets
print("\nStep 12: Saving preprocessed datasets")

# Create output directory if needed
import os
output_dir = '../data/preprocessed'
os.makedirs(output_dir, exist_ok=True)

# Save training data
X_train_scaled.to_csv(f'{output_dir}/X_train.csv', index=False)
y_train_smote.to_csv(f'{output_dir}/y_train.csv', index=False)

# Save validation data
X_val_scaled.to_csv(f'{output_dir}/X_val.csv', index=False)
y_val.to_csv(f'{output_dir}/y_val.csv', index=False)

# Save test data
X_test_scaled.to_csv(f'{output_dir}/X_test.csv', index=False)
y_test.to_csv(f'{output_dir}/y_test.csv', index=False)

# Save scaler for later use
import pickle
with open(f'{output_dir}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f"Saved preprocessed datasets to {output_dir}/")
print(f"  X_train.csv: {X_train_scaled.shape}")
print(f"  y_train.csv: {y_train_smote.shape}")
print(f"  X_val.csv: {X_val_scaled.shape}")
print(f"  y_val.csv: {y_val.shape}")
print(f"  X_test.csv: {X_test_scaled.shape}")
print(f"  y_test.csv: {y_test.shape}")
print(f"  scaler.pkl: StandardScaler object")

print("\n✓ Data preprocessing complete!")


Step 12: Saving preprocessed datasets
Saved preprocessed datasets to ../data/preprocessed/
  X_train.csv: (91314, 2343)
  y_train.csv: (91314,)
  X_val.csv: (10728, 2343)
  y_val.csv: (10728,)
  X_test.csv: (10728, 2343)
  y_test.csv: (10728,)
  scaler.pkl: StandardScaler object

✓ Data preprocessing complete!


## Preprocessing Summary

### Data Transformations Applied:
✓ Dropped duplicate patient encounters (kept first per patient)  
✓ Dropped weight & payer_code columns (excessive missing values)  
✓ Dropped columns with >90% missing values  
✓ Created binary target: readmitted within 30 days (0/1)  
✓ Mapped admission_type_id using IDS_mapping.csv  
✓ Mapped discharge_disposition_id using IDS_mapping.csv  
✓ Mapped admission_source_id using IDS_mapping.csv  
✓ Encoded age as ordinal values (0-9 for age brackets)  
✓ Encoded 23 medication columns as change indicators (0=no change, 1=change)  
✓ One-hot encoded remaining categorical variables  
✓ Applied SMOTE to training data to handle class imbalance  
✓ Scaled all numeric features using StandardScaler  
✓ Created 70/15/15 train/validation/test split with stratification  

### Final Dataset Shapes:
- Training: {X_train_scaled.shape[0]} samples × {X_train_scaled.shape[1]} features
- Validation: {X_val_scaled.shape[0]} samples × {X_val_scaled.shape[1]} features
- Test: {X_test_scaled.shape[0]} samples × {X_test_scaled.shape[1]} features

All preprocessed data saved to `data/preprocessed/` directory.